1. If you have trained five different models on the exact same training data, and
they all achieve 95% precision, is there any chance that you can combine these
models to get better results? If so, how? If not, why?

- Щом има висок precision се предполага че ще има нисък recall и ако ги комбинираме би могло да се направят по добри предикшани така че цялостното f1 да се подобри
- Но ако моделите правят еднакви грешки комбиниране с hard voting няма да помогне, а по добра идея ще е да се използва някой от boosting алгоритмите 

2. What is the difference between hard and soft voting classifiers?

- при hard - гласуване чрез мнозинство, взима се класа който е бил предскзан от най много естиматори
- при soft - взима се вероятностите от моделите и се осредняват на класовете тоест се взима класа за който моделите са били най сигурни

3. Is it possible to speed up training of a bagging ensemble by distributing it across
multiple servers? What about pasting ensembles, boosting ensembles, random
forests, or stacking ensembles?

- Да, паралелното обучение върху няколко ядра ще бъде по бързо
- pasting ensembles - Да, пак ще се подобри
- boosting ensembles - при повечето обучението не може да бъде парално и е последованетолно като при AdaBoost, GradientBoosting
- random forests - Да, пак ще се подобри
- stacking ensembles - Да, пак ще се подобри

4. What is the benefit of out-of-bag evaluation?

- предимството е че няма да има нужда предварително да си заделяме валидационен сет а по време на обучението като се използва bootstrap данните които не са били подадени на модела за да се обучи с тях се използат за да направи oof предикшан и така като се съберат от различните модели предикшаните се агрегират и се оценяват с оригиналните данни и така получаваме производителността на ансамбъла

5. What makes extra-trees ensembles more random than regular random forests?
How can this extra randomness help? Are extra-trees classifiers slower or faster
than regular random forests?

- той за всяко дързо за всеки негов възел използва няколко случайно избрани характеристики и вместо като random forests да го сортира и да търси най добрия праг той избира един на случаен принцип в диапазона на характеристиката и след това избина за конкретния възел най добрия сплит
- по този начин на дърветата се увеличава bias но като цяло на целия ансамбъл variance спада защото дърветата са много различни и се компенсират
- след като за всяко дърво и негов възел се пропуска това да се тързи най добрия праг то го прави многократно по бърз от random forest 

6. If your AdaBoost ensemble underfits the training data, which hyperparameters
should you tweak, and how?

- бих увеличил n_estimators и ще намаля малко learning_rate

7. If your gradient boosting ensemble overfits the training set, should you increase
or decrease the learning rate?

- ще намаля learning rate защото така ще намаля влиянието на последователно предсказаните грешки от миделите

8. Load the MNIST dataset (introduced in Chapter 3), and split it into a training
set, a validation set, and a test set (e.g., use 50,000 instances for training, 10,000
for validation, and 10,000 for testing). Then train various classifiers, such as a
random forest classifier, an extra-trees classifier, and an SVM classifier. Next, try
to combine them into an ensemble that outperforms each individual classifier
on the validation set, using soft or hard voting. Once you have found one, try
it on the test set. How much better does it perform compared to the individual
classifiers?

In [2]:
from sklearn.datasets import fetch_openml
import numpy as np
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd

In [5]:
mnist = fetch_openml('mnist_784', as_frame=False) 

X, y = mnist.data, mnist.target

np.random.seed(42)
shuffle_inxs = np.random.permutation(X.shape[0])
X, y = X[shuffle_inxs], y[shuffle_inxs]

X_train, X_val, X_test, y_train, y_val, y_test = X[:50000], X[50000:60000], X[60000:], y[:50000], y[50000:60000], y[60000:]

In [5]:
X.shape

(70000, 784)

Train RandomForest

In [12]:
forest = RandomForestClassifier(random_state=42, n_jobs=-1).fit(X_train, y_train)

y_test_pred = forest.predict(X_test)
forest_acc = accuracy_score(y_test, y_test_pred)
forest_f1 = f1_score(y_test, y_test_pred, average='micro')

print('Forest ACC:', forest_acc)
print('Forest F1:', forest_f1)

Forest ACC: 0.9684
Forest F1: 0.9684


Train ExtraTreesClassifier

In [13]:
extra_tree = ExtraTreesClassifier(random_state=42, n_jobs=-1).fit(X_train, y_train)

y_test_pred = extra_tree.predict(X_test)
extra_tree_acc = accuracy_score(y_test, y_test_pred)
extra_tree_f1 = f1_score(y_test, y_test_pred, average='micro')

print('ExtraTrees ACC:', extra_tree_acc)
print('ExtraTrees F1:', extra_tree_f1)

ExtraTrees ACC: 0.9711
ExtraTrees F1: 0.9711


Train SVC

In [ ]:
svc = SVC(random_state=42).fit(X_train, y_train)

y_test_pred = svc.predict(X_test)
svc_acc = accuracy_score(y_test, y_test_pred)
svc_f1 = f1_score(y_test, y_test_pred, average='micro')

print('SVC ACC:', svc_acc)
print('SVC F1:', svc_f1)

SVC ACC: 0.9776
SVC F1: 0.9776


Train Voting

In [9]:
est_list = [
    ('forest', RandomForestClassifier(random_state=42, n_jobs=-1)),
    ('extra_trees', ExtraTreesClassifier(random_state=42, n_jobs=-1)),
    ('svc', SVC(random_state=42, probability=True))   
]

Test Hard Voting

In [7]:
hard_voting = VotingClassifier(est_list, voting='hard', n_jobs=-1).fit(X_train, y_train)

y_val_pred = hard_voting.predict(X_val)
hard_voting_val_acc = accuracy_score(y_val, y_val_pred)
hard_voting_val_f1 = f1_score(y_val, y_val_pred, average='micro')

print('Hard Voting ACC:', hard_voting_val_acc)
print('Hard Voting F1:', hard_voting_val_f1)

Hard Voting ACC: 0.9733
Hard Voting F1: 0.9733


Test Soft Voting

In [10]:
soft_voting = VotingClassifier(est_list, voting='soft', n_jobs=-1).fit(X_train, y_train)

y_val_pred = soft_voting.predict(X_val)
soft_voting_val_acc = accuracy_score(y_val, y_val_pred)
soft_voting_val_f1 = f1_score(y_val, y_val_pred, average='micro')

print('Hard Voting ACC:', soft_voting_val_acc)
print('Hard Voting F1:', soft_voting_val_f1)

Hard Voting ACC: 0.978
Hard Voting F1: 0.978


In [11]:
y_test_pred = soft_voting.predict(X_test)
soft_voting_test_acc = accuracy_score(y_test, y_test_pred)
soft_voting_test_f1 = f1_score(y_test, y_test_pred, average='micro')

print('Hard Voting ACC:', soft_voting_test_acc)
print('Hard Voting F1:', soft_voting_test_f1)

Hard Voting ACC: 0.9779
Hard Voting F1: 0.9779


In [14]:
rows = ['Forest', 'ExtraTrees', 'SVC', 'SoftVot']
data = [
    [forest_acc, forest_f1],
    [extra_tree_acc, extra_tree_f1],
    [svc_acc, svc_f1],
    [soft_voting_test_acc, soft_voting_test_f1]
]

results = pd.DataFrame(data, rows, ['ACC', 'F1'])
results

,ACC,F1
Forest,0.9684,0.9684
ExtraTrees,0.9711,0.9711
SVC,0.9776,0.9776
SoftVot,0.9779,0.9779


9. Run the individual classifiers from the previous exercise to make predictions on
the validation set, and create a new training set with the resulting predictions:
each training instance is a vector containing the set of predictions from all your
classifiers for an image, and the target is the image’s class. Train a classifier
on this new training set. Congratulations—you have just trained a blender, and together with the classifiers it forms a stacking ensemble! Now evaluate the
ensemble on the test set. For each image in the test set, make predictions with all
your classifiers, then feed the predictions to the blender to get the ensemble’s pre‐
dictions. How does it compare to the voting classifier you trained earlier? Now
try again using a StackingClassifier instead. Do you get better performance? If
so, why?

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier

In [18]:
def get_blender_X(base_est: list, X):
    preds = []
    for est in base_est:
        y_pred = est.predict(X)
        if y_pred.ndim == 1:
            y_pred = y_pred.reshape(-1, 1)

        preds.append(y_pred)

    return np.hstack(preds)

base_est_list = [forest, extra_tree, svc]
blender = LogisticRegression(random_state=42, n_jobs=-1)

In [19]:
X_blender = get_blender_X(base_est_list, X_val)
blender.fit(X_blender, y_val)

/home/zdravko/Machine_Learning_Intern/venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [20]:
X_test_blender = get_blender_X(base_est_list, X_test)
y_blender_pred = blender.predict(X_test_blender)

custom_stacking_acc = accuracy_score(y_test, y_blender_pred)
custom_stacking_f1 = f1_score(y_test, y_blender_pred, average='micro')

print('Custom Stacking Test ACC:', custom_stacking_acc)
print('Custom Stacking Test F1:', custom_stacking_f1)

Custom Stacking Test ACC: 0.9683
Custom Stacking Test F1: 0.9683


In [6]:
stack = StackingClassifier([
    ('forest', RandomForestClassifier(random_state=42, n_jobs=-1)),
    ('extratrees', ExtraTreesClassifier(random_state=42, n_jobs=-1)),
    ('svc', SVC(random_state=42))
], final_estimator=LogisticRegression(random_state=42, n_jobs=-1), cv=3)

stack.fit(X_train, y_train)

y_stack_pred = stack.predict(X_test)
stack_test_acc = accuracy_score(y_test, y_stack_pred)
stack_test_f1 = f1_score(y_test, y_stack_pred, average='micro')

print('Stacking Test ACC:', stack_test_acc)
print('Stacking Test F1:', stack_test_f1)

/home/zdravko/Machine_Learning_Intern/venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Stacking Test ACC: 0.9777
Stacking Test F1: 0.9777


Оригиналния стакинг има по добро представяне защото по време на неговото обучение той обучава блендера на oof предстаказания на базовоте естиматори на целия тренировъчен сет а моя блендер е обучен на предсказанията от валидационния сет и освен това не използва директно техните предикшани както моя, а се опитва да използва predict_proba или decision_function което му подобрява представянето